In [20]:
import numpy
import math
import json

In [8]:
#----------------------------------------------------------------------------
#-- sort_multi_lists: sort all lists in list_save with in the sorted order of the list at postion given
#----------------------------------------------------------------------------

def sort_multi_lists(array_save, pos=0):
    """
    sort all lists in list_save with in the sorted order of the list at postion given
    input:  array_save  --- a list of lists
            pos         --- the position of the list to be ordered
    output: save        --- a list of lists ordered 
    """

    idx    = numpy.argsort(array_save[pos])
    save   = []
    for tarray in array_save:

        save.append(tarray[idx])

    return save

In [9]:
def mapsphere(distmapmax,xinc,yinc,zinc):
    """
    this routine finds the (i,j,k) index offset values which are
    used to define the search volume for the near-neighbor flux
    search.
    
    input:  distmapmax --- maximum distance from flux measurement location
                            allowed for streamline mapping (re).
            xinc       --- extent of database cell in x-direction (re).
            yinc       --- extent of database cell in y-direction (re).
            zinc       --- extent of database cell in z-direction (re).
    
    output: nsphvol    --- number of volume elements stored in the
                            database search volume.
            ioffset    --- array of offset indices for x-direction.
            joffset    --- array of offset indices for y-direction.
            koffset    --- array of offset indices for z-direction.
    
    *** note *** the indices are sorted by range from the center of
    the spherical search volume (ascending order on range).

    """
#
#--- get the number of volume bins that extends 1/2 the length
#--- of each side of the cube which contains the spherical search
#--- volume defined by distmapmax
#
    istep   = int(distmapmax / xinc)
    jstep   = int(distmapmax / yinc)
    kstep   = int(distmapmax / zinc)
#
#--- store the index offset values for each volume element that
#--- lies within the search volume sphere
#
    istart    = -istep
    jstart    = -jstep
    kstart    = -kstep
    istop     =  istep + 1
    jstop     =  jstep + 1
    kstop     =  kstep + 1

    rngoffset = numpy.zeros(maxnsphvol)
    ioffset   = numpy.zeros(maxnsphvol).astype(int)
    joffset   = numpy.zeros(maxnsphvol).astype(int)
    koffset   = numpy.zeros(maxnsphvol).astype(int)
    nsphvol   =0

    for i in range(istart, istop):
        for j in range(jstart, jstop):
            for k in range(kstart, kstop):
#
#--- calculate the distance of this volume element from the
#--- search volume's central volume element
#
                xrng = i * xinc
                yrng = j * yinc
                zrng = k * zinc
                dist = math.sqrt(xrng**2 + yrng**2 + zrng**2)
#
#--- store this volume element's indices as part of the spherical search volume
#
                if dist <= distmapmax:
                    rngoffset[nsphvol] = dist
                    ioffset[nsphvol] = i
                    joffset[nsphvol] = j
                    koffset[nsphvol] = k
                    nsphvol += 1
#
#--- sort the offset index arrays in ascending by geocentric range
#--- from the center of the search volume
#
    rngoffset  = rngoffset[:nsphvol]
    ioffset    = ioffset[:nsphvol]
    joffset    = joffset[:nsphvol]
    koffset    = koffset[:nsphvol]
    [rngoffset, ioffset, joffset, koffset] =\
                    sort_multi_lists([rngoffset, ioffset, joffset, koffset])

    return nsphvol, ioffset, joffset, koffset

In [12]:
distmapmax3 = 20.0
xinc    = 1.0       #--- Chandra Volume Element Database Parameters: length of volume element in x-direction (Re)
yinc    = 1.0       #--- Chandra Volume Element Database Parameters: length of volume element in y-direction (Re)
zinc    = 1.0       #--- Chandra Volume Element Database Parameters: length of volume element in z-direction (Re)
maxnsphvol = 100000 #--- maximum number of sub-volume elements stores in the streamline mapping search volume

In [13]:
nsphvol3, ioffset3,joffset3,koffset3 = mapsphere(distmapmax3,xinc,yinc,zinc)

In [37]:
_dict = {
    'nsphvol3':nsphvol3,
    'ioffset3':ioffset3.tolist(),
    'joffset3':joffset3.tolist(),
    'koffset3':koffset3.tolist()
}

In [38]:
with open('index_offset.json','w') as f:
    json.dump(_dict,f)